# 01 · EDA – Home Credit Default Risk
Data quality, distributions, missingness. Findings are summarised in `docs/eda.md`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parents[1] if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
RAW = ROOT / "data/raw/home-credit"

In [ ]:
app = pd.read_csv(RAW / "application_train.csv")
app.shape, app.TARGET.mean()

## Target imbalance
~8% defaults → ROC-AUC/PR-AUC instead of accuracy; class weights + calibration.

In [ ]:
app.TARGET.value_counts(normalize=True).plot.bar(title="TARGET share"); plt.show()

## Missing values

In [ ]:
miss = app.isna().mean().sort_values(ascending=False)
print((miss > .5).sum(), "columns > 50% missing")
plt.figure(figsize=(14, 4)); plt.imshow(app.sample(2000, random_state=0)[miss.index[:60]].isna().T, aspect="auto", cmap="Greys")
plt.yticks(range(60), miss.index[:60], fontsize=5); plt.title("missingness (rows x top-60 columns)"); plt.show()

## Known anomaly: `DAYS_EMPLOYED = 365243`
18% of rows (pensioners / unemployed). Replaced by NaN + `DAYS_EMPLOYED_ANOM` flag in `features/application.py`.

In [ ]:
anom = app.DAYS_EMPLOYED == 365243
anom.mean(), app.TARGET[anom].mean(), app.TARGET[~anom].mean()

## Distributions by target

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, c in zip(axes, ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"]):
    for t in (0, 1):
        app.loc[app.TARGET == t, c].plot.kde(ax=ax, label=f"TARGET={t}")
    ax.set_title(c); ax.legend()
plt.show()

## Related tables

In [ ]:
for f in ["bureau", "bureau_balance", "previous_application", "installments_payments", "POS_CASH_balance", "credit_card_balance"]:
    df = pd.read_csv(RAW / f"{f}.csv", nrows=200_000)
    print(f"{f:25s}", df.shape[1], "cols  example columns:", list(df.columns[:6]))